# Very Simple ETL Pipeline: Crypto Coins

This notebook shows a beginner-friendly ETL workflow using:

- `requests` to **extract** data from a public API
- `.json()` to convert the API response into Python data
- `pandas` to **transform** the data into a table
- `.to_csv()` to **load** / save the result
- `matplotlib` to create simple charts

We will use the public CoinGecko API. No password or API key is needed for this simple example.

## 1. Import libraries

In [16]:
import requests
import pandas as pd
import os

## 2. EXTRACT: load Pokémon data

In [17]:
pokemon_names = [
    "pikachu", "bulbasaur", "charmander", "squirtle", "jigglypuff",
    "meowth", "psyduck", "machop", "eevee", "snorlax"
]

pokemon_data = []

for name in pokemon_names:
    url = f"https://pokeapi.co/api/v2/pokemon/{name}"
    response = requests.get(url)
    response.raise_for_status()
    pokemon_data.append(response.json())

print("Number of Pokemon loaded:", len(pokemon_data))

Number of Pokemon loaded: 10


## 3. TRANSFORM: convert JSON to table

In [19]:
rows = []

for p in pokemon_data:
    rows.append({
        "pokemon_name": p["name"],
        "height_dm": p["height"],
        "weight_hg": p["weight"],
        "base_experience": p["base_experience"],
        "main_type": p["types"][0]["type"]["name"],
        "first_ability": p["abilities"][0]["ability"]["name"]
    })

df = pd.DataFrame(rows)

print("Raw table shape:", df.shape)
df.head()

Raw table shape: (10, 6)


,pokemon_name,height_dm,weight_hg,base_experience,main_type,first_ability
0,pikachu,4,60,112,electric,static
1,bulbasaur,7,69,64,grass,overgrow
2,charmander,6,85,62,fire,blaze
3,squirtle,5,90,63,water,torrent
4,jigglypuff,5,55,95,normal,cute-charm


## 4. TRANSFORM: create new columns

In [20]:
df["height_m"] = df["height_dm"] / 10
df["weight_kg"] = df["weight_hg"] / 10
df["bmi_like_metric"] = df["weight_kg"] / (df["height_m"] ** 2)

df.head()

,pokemon_name,height_dm,weight_hg,base_experience,main_type,first_ability,height_m,weight_kg,bmi_like_metric
0,pikachu,4,60,112,electric,static,0.4,6.0,37.500000
1,bulbasaur,7,69,64,grass,overgrow,0.7,6.9,14.081633
2,charmander,6,85,62,fire,blaze,0.6,8.5,23.611111
3,squirtle,5,90,63,water,torrent,0.5,9.0,36.000000
4,jigglypuff,5,55,95,normal,cute-charm,0.5,5.5,22.000000


## 5. TRANSFORM: select, rename, sort

In [21]:
df_clean = df[
    [
        "pokemon_name",
        "main_type",
        "first_ability",
        "height_m",
        "weight_kg",
        "base_experience",
        "bmi_like_metric"
    ]
].copy()

df_clean = df_clean.rename(columns={
    "pokemon_name": "name",
    "main_type": "type",
    "first_ability": "ability",
    "base_experience": "base_xp"
})

df_clean = df_clean.sort_values("base_xp", ascending=False)

df_clean

,name,type,ability,height_m,weight_kg,base_xp,bmi_like_metric
9,snorlax,normal,immunity,2.1,460.0,189,104.308390
0,pikachu,electric,static,0.4,6.0,112,37.500000
4,jigglypuff,normal,cute-charm,0.5,5.5,95,22.000000
8,eevee,normal,run-away,0.3,6.5,65,72.222222
1,bulbasaur,grass,overgrow,0.7,6.9,64,14.081633
6,psyduck,water,damp,0.8,19.6,64,30.625000
3,squirtle,water,torrent,0.5,9.0,63,36.000000
2,charmander,fire,blaze,0.6,8.5,62,23.611111
7,machop,fighting,guts,0.8,19.5,61,30.468750
5,meowth,normal,pickup,0.4,4.2,58,26.250000


## 6. LOAD: save processed CSV

In [22]:
os.makedirs("data/processed", exist_ok=True)

df_clean.to_csv("data/processed/output.csv", index=False)

print("Saved file: data/processed/output.csv")

Saved file: data/processed/output.csv


## 7. Check output

In [23]:
output = pd.read_csv("data/processed/output.csv")

print("Output shape:", output.shape)
output.head()

Output shape: (10, 7)


,name,type,ability,height_m,weight_kg,base_xp,bmi_like_metric
0,snorlax,normal,immunity,2.1,460.0,189,104.308390
1,pikachu,electric,static,0.4,6.0,112,37.500000
2,jigglypuff,normal,cute-charm,0.5,5.5,95,22.000000
3,eevee,normal,run-away,0.3,6.5,65,72.222222
4,bulbasaur,grass,overgrow,0.7,6.9,64,14.081633
